# Remove Redundant Comparison Links
Identifies and removes unnecessary directed comparison links from the database.

## Redundancy Rule
A directed link `u → v` (u beat v) is **unnecessary** if a longer directed chain `u → ... → v` (length ≥2) exists. The direct link is redundant because the longer chain already establishes u is better than v.

## Examples
1. Chain: `a → b → c → d → e`, extra link `a → d` → **Redundant** (path `a→b→c→d` exists)
2. Edges: `a→b, b→c, d→c (d beat c), a→d` → **Not redundant** (no longer directed path from a to d)

In [ ]:
import os
import sys
from pathlib import Path

# Add plugin root to path
notebook_path = Path(os.getcwd()).resolve()
plugin_root = notebook_path.parents[1]
print(f"Plugin root: {plugin_root}")

if str(plugin_root) not in sys.path:
    sys.path.insert(0, str(plugin_root))
if str(notebook_path) not in sys.path:
    sys.path.insert(0, str(notebook_path))

print(f"Current working directory: {os.getcwd()}")


Plugin root: E:\ComfyUI\custom_nodes\comfyui-image-scorer
Current working directory: e:\ComfyUI\custom_nodes\comfyui-image-scorer\external_modules\step01ranking_new


In [ ]:
from tqdm.notebook import tqdm

# Import CrystalGraph singleton (already builds graph on import)
from shared.graph.crystal_graph import crystal_graph

# Import delete function
from database.comparisons_table import delete_comparison

print("CrystalGraph loaded. Graph built at:", crystal_graph._built_at)


CrystalGraph loaded. Graph built at: 2026-05-04 14:59:41.781843+00:00


In [ ]:
# --- CONFIGURATION ---
DRY_RUN = False  # Set to False to apply actual changes
DELETE_COMPARISONS = True  # Set to True to delete redundant links from DB
# Both DRY_RUN=False and DELETE_COMPARISONS=True required to make changes
# ---------------------


In [ ]:
print("Using prebuilt CrystalGraph from shared module...")
print(f"Total images: {len(crystal_graph.images)}")
print(f"Total comparisons: {len(crystal_graph.comparisons)}")
print(
    f"Directed edges (winner->loser): {sum(len(v) for v in crystal_graph.worse.values())}"
)


Using prebuilt CrystalGraph from shared module...
Total images: 26770
Total comparisons: 18234
Directed edges (winner->loser): 18234


In [ ]:
print("Identifying redundant edges...")
redundant_edges = []
necessary_edges = []

# Iterate over all directed edges (winner -> loser)
for u, losers in crystal_graph.worse.items():
    for v in losers:
        if crystal_graph.is_redundant(u, v):
            redundant_edges.append((u, v))
        else:
            necessary_edges.append((u, v))

print(f"\nTotal directed edges: {len(redundant_edges) + len(necessary_edges)}")
print(f"Redundant edges (to remove): {len(redundant_edges)}")
print(f"Necessary edges (to keep): {len(necessary_edges)}")


Identifying redundant edges...

Total directed edges: 18234
Redundant edges (to remove): 0
Necessary edges (to keep): 18234


In [ ]:
if redundant_edges:
    print("\nExample redundant edges (u → v):")
    for u, v in redundant_edges[:5]:
        print(f"  {u} → {v}")
else:
    print("\nNo redundant edges found.")

if necessary_edges:
    print("\nExample necessary edges (u → v):")
    for u, v in necessary_edges[:5]:
        print(f"  {u} → {v}")



No redundant edges found.

Example necessary edges (u → v):
  00343_random--cyberrealisticPony_v160.png → 00425_random--cyberrealisticPony_v160.png
  00343_random--cyberrealisticPony_v160.png → 00202_random-shirtliftv1-cyberrealistic_v90.png
  00417_random-add_detail-SD15_realisticVisionV60B1_v51HyperVAE.png → 00174_random-shirtliftv1-SD15_lazymixRealAmateur_v40.png
  00123_random--realismByStableYogi_ponyV3VAE.png → 00186_random-PSCowgirl-realisticVisionV60B1_v51HyperVAE.png
  00418_random-shirtliftv1-SD15_dreamshaper_8.png → 00107_random-add_detail-realisticVisionV60B1_v51HyperVAE.png


In [ ]:
if DELETE_COMPARISONS and not DRY_RUN:
    print(f"\nDeleting {len(redundant_edges)} redundant edges from database...")
    deleted = 0
    errors = 0
    for u, v in tqdm(redundant_edges, desc="Deleting"):
        # u is winner, v is loser; delete_comparison handles canonicalization
        result = delete_comparison(u, v, u)
        if result > 0:
            deleted += 1
        else:
            errors += 1
    print(f"Deleted {deleted} comparison records. Errors: {errors}")
elif DRY_RUN and DELETE_COMPARISONS:
    print(f"\n[DRY RUN] Would delete {len(redundant_edges)} redundant edges.")
else:
    print("\nNo changes made (DELETE_COMPARISONS=False or DRY_RUN=True).")



Deleting 0 redundant edges from database...


Deleting: 0it [00:00, ?it/s]

Deleted 0 comparison records. Errors: 0
